# Transcription-factor activity along thymic ageing, scored on differential-expression statistics — SCENIC+ eRegulons

The companion of `01_CollecTRI_TF_activity_on_ageDE.ipynb`: the same DESeq2 Wald statistics, the same
univariate-linear-model scoring and abundance-matched permutation null, applied to the **thymus
enhancer-driven regulons (eRegulons) inferred by SCENIC+ from the matched multiome data** instead of
a literature network. The notebook is self-contained and produces:

1. the per-cell-type × eRegulon score table (Supplementary Table);
2. violin plots placing the steroid-receptor eRegulons within the distribution of all eRegulons, in
   the stroma and across all cell types (only AR and NR3C1 have an eRegulon in this network);
3. the 20 eRegulons most consistently up- and down-regulated across the stromal populations.

One property of the eRegulon network needs handling (section 4b): every eRegulon is either
activating or repressing, and the two classes sit on opposite sides of zero for a structural reason,
so regulons are ranked **within their class** (`pct_z_mode`).

Inputs: the DESeq2 tables, `data/thyAgeing_celltype_levels.csv`, and the eRegulon network
`data/ThyAgeing_SCENICplus_eRegulons_decoupler_net_signonly.csv`. Runtime ≈ 2 min on 4 cores (1,000 permutations).


In [1]:
import os, re, sys, time, warnings
import numpy as np
import pandas as pd
import decoupler as dc
from scipy.stats import norm, spearmanr, percentileofscore
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_rgb
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print(f'python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | '
      f'decoupler {dc.__version__} | matplotlib {matplotlib.__version__} | seaborn {sns.__version__}')


python 3.12.0 | numpy 2.2.4 | pandas 2.2.3 | decoupler 2.1.1 | matplotlib 3.10.1 | seaborn 0.13.2


In [2]:
# ------------------------------------------------------------------ paths (edit for your setup)
DEA_DIR = '/nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/data/analyses/dea'  # DESeq2 tables
HERE    = os.environ.get('NB_OUT_DIR', os.getcwd())          # this folder: data/, results/, figures/
DATA_DIR, RESULTS, FIGURES = f'{HERE}/data', f'{HERE}/results', f'{HERE}/figures'
for d in (RESULTS, FIGURES):
    os.makedirs(d, exist_ok=True)
NET_CSV = f'{DATA_DIR}/ThyAgeing_SCENICplus_eRegulons_decoupler_net_signonly.csv'

# ------------------------------------------------------------------ analysis settings
LEVEL       = 'taa_l2_v2'
CONTRASTS   = ['paed_vs_infant', 'adult_vs_infant']
NET_NAME    = 'SCENICplus'
PREFIX      = f'thyAgeing_{NET_NAME}_ageDE_{LEVEL}'
STEROID_TFS = ['AR', 'NR3C1', 'PGR', 'ESR1', 'ESR2']
STROMA_L1   = ['TEC', 'Fb', 'EC', 'Mural']

TMIN     = 5
STAT_CAP = 20.0
N_BINS   = 20
N_PERM   = int(os.environ.get('NB_N_PERM', 1000))
SEED     = 0
FORCE_RESCORE = os.environ.get('NB_FORCE_RESCORE', '0') == '1'

SCORES_CSV = f'{RESULTS}/{PREFIX}_scores_perCellType.csv'
print(f'{NET_NAME} | {LEVEL} | outputs in {HERE}')


SCENICplus | taa_l2_v2 | outputs in /nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/notebooks/hormone2cell/_builders/../final_versions/github


In [3]:
# ------------------------------------------------------------------ figure style
import logging
logging.getLogger('fontTools').setLevel(logging.WARNING)   # PDF font subsetting is chatty at INFO
logging.getLogger('matplotlib').setLevel(logging.WARNING)
try:                                                    # Arial if available (Nature house font)
    font_manager.fontManager.addfont('/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf')
except Exception:
    pass
rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'pdf.fonttype': 42, 'ps.fonttype': 42,             # text stays editable in Illustrator
    'font.size': 7, 'axes.titlesize': 8, 'axes.labelsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7, 'figure.titlesize': 8,
    'axes.linewidth': 0.5, 'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2, 'xtick.major.pad': 1.5, 'ytick.major.pad': 1.5,
    'savefig.dpi': 300, 'savefig.bbox': 'standard',    # 'standard': the PDF keeps the size in mm below
    'savefig.transparent': False, 'figure.dpi': 110,
})
MM = 1 / 25.4                                            # inches per mm
NATURE_W = {'single': 89.0, 'two-thirds': 120.0, 'double': 183.0}   # Nature figure widths, mm
MISSING = '#e3e0e0'                                      # cell type not tested in a contrast
DIVERGING = sns.color_palette('vlag', as_cmap=True).copy()
DIVERGING.set_bad(MISSING)

# One colour + marker per steroid receptor (identical across all figures of the manuscript).
RECEPTOR_STYLE = {'AR':    ('#d95f02', 'o'),
                  'NR3C1': ('#1b9e77', '*'),
                  'PGR':   ('#7570b3', 'D'),
                  'ESR1':  ('#b2182b', 's'),
                  'ESR2':  ('#56B4E9', '^')}


def mm_fig(width_mm, height_mm):
    """A figure of exactly this size in millimetres; constrained layout keeps labels inside it."""
    return plt.figure(figsize=(width_mm * MM, height_mm * MM), layout='constrained')


def save_fig(fig, name):
    """Save `FIGURES/<name>.pdf` at the figure's size in mm, and report it."""
    path = f'{FIGURES}/{name}.pdf'
    fig.savefig(path)
    w, h = np.asarray(fig.get_size_inches()) * 25.4
    print(f'saved {os.path.basename(path)}  ({w:.0f} x {h:.0f} mm)')
    return path


In [4]:
# ------------------------------------------------------------------ cell-type order and lineage
# `thyAgeing_celltype_levels.csv` is the curated annotation hierarchy of the atlas (one row per
# finest label; columns taa_l1, taa_l2_v2, taa_l3, ... in the atlas' display order). It fixes the
# order of cell types in every figure and defines the lineages: "stroma" = TEC + Fb + EC + Mural.
CT_LEVELS = pd.read_csv(f'{DATA_DIR}/thyAgeing_celltype_levels.csv', dtype=str)
assert LEVEL in CT_LEVELS.columns, f'{LEVEL} is not a column of thyAgeing_celltype_levels.csv'
LINEAGE = (CT_LEVELS[[LEVEL, 'taa_l1']].dropna().drop_duplicates(LEVEL)
           .set_index(LEVEL)['taa_l1'].to_dict())          # cell type -> broad lineage (taa_l1)


def ct_order(lineages=None, present=None, level=LEVEL):
    """Cell types of `level` in atlas order, restricted to taa_l1 `lineages` and/or to `present`."""
    d = CT_LEVELS if lineages is None else CT_LEVELS[CT_LEVELS['taa_l1'].isin(lineages)]
    cts = list(dict.fromkeys(d[level].dropna()))
    return cts if present is None else [c for c in cts if c in set(present)]


# lineage colours: one distinct hue per broad lineage (stromal lineages first; the immune lineages
# use hues that do not collide with them), so points of different lineages never share a colour
_lin_all = list(dict.fromkeys(CT_LEVELS['taa_l1'].dropna()))
LINEAGE_COLORS = {'TEC': '#0072B2', 'Fb': '#E69F00', 'EC': '#009E73', 'Mural': '#CC79A7',
                  'T': '#D62728', 'NK': '#9467BD', 'B': '#8C564B', 'Mono': '#7F7F7F',
                  'Mac': '#BCBD22', 'DC': '#17BECF', 'Myeloid_dev': '#000000',
                  'Adipocyte': '#FF9896', 'Neutrophil': '#C5B0D5', 'Mast': '#C49C94'}
_extra = sns.color_palette('Set2', n_colors=8)                    # fallback for any other lineage
for i, l in enumerate([l for l in _lin_all if l not in LINEAGE_COLORS]):
    LINEAGE_COLORS[l] = matplotlib.colors.to_hex(_extra[i % len(_extra)])
print(f'{len(_lin_all)} lineages (taa_l1): {", ".join(_lin_all)}')
print(f'{CT_LEVELS[LEVEL].nunique()} cell types at {LEVEL}; stroma = {ct_order(lineages=STROMA_L1)}')


14 lineages (taa_l1): T, NK, TEC, Fb, Adipocyte, EC, Mural, B, Myeloid_dev, Neutrophil, Mono, Mac, DC, Mast
30 cell types at taa_l2_v2; stroma = ['cTEC', 'mcTEC', 'mTEC', 'TEC-EMT', 'Fb', 'EC-blood', 'EC-lymphatic', 'SMC', 'Pericyte']


## 1. Input: per-cell-type differential expression statistics

For every cell type, DESeq2 was run on donor-level pseudobulk profiles with age group as the
covariate and *infant* as the reference; the two contrasts used here are **paediatric vs infant**
and **adult vs infant** (see the differential-expression methods). The quantity scored below is the
**Wald statistic** (`stat`) of every gene DESeq2 tested in that cell type, together with the gene's
mean normalised expression (`baseMean`), which the permutation null is matched on.

Why the Wald statistic rather than expression: a footprint score computed on expression tracks the
*abundance* of a regulon's targets (highly expressed regulons score high whether or not anything
changes). A Wald statistic is a contrast centred on zero, so abundance cannot masquerade as
activity. The permutation null below removes the residual abundance dependence of the score.

The statistic is capped at ±20 so that a degenerate DESeq2 fit (`lfcSE → 0`) cannot set the scale of
its whole cell type.


In [5]:
DE_COLS = ['cell_type', 'gene_name', 'baseMean', 'stat']


def read_de_table(path):
    """One contrast: flat `cell_type, gene_name, baseMean, stat`. The gene column is `gene_name` or `gene`."""
    head = pd.read_csv(path, nrows=0).columns
    gene_col = 'gene_name' if 'gene_name' in head else 'gene'
    return (pd.read_csv(path, usecols=['cell_type', gene_col, 'baseMean', 'stat'])
              .rename(columns={gene_col: 'gene_name'})[DE_COLS])


def load_de(level, contrasts, dea_dir=DEA_DIR, stat_cap=STAT_CAP):
    """Both contrasts in one long table, Wald statistic capped at +/- stat_cap."""
    de = pd.concat([read_de_table(f'{dea_dir}/thyAgeing_dea_{level}_{c}_ageEffect.csv').assign(contrast=c)
                    for c in contrasts], ignore_index=True)
    assert np.isfinite(de['stat']).all(), 'non-finite Wald statistics in the DE tables'
    hit = de.loc[de['stat'].abs() > stat_cap, ['cell_type', 'contrast', 'gene_name', 'baseMean', 'stat']]
    print(f'{len(hit)} of {len(de):,} Wald statistics capped at +/-{stat_cap:g}')
    if len(hit):
        display(hit)
    de['stat'] = de['stat'].clip(-stat_cap, stat_cap)
    return de


de = load_de(LEVEL, CONTRASTS)

ct_by_contrast = {c: sorted(de.loc[de['contrast'] == c, 'cell_type'].unique()) for c in CONTRASTS}
shared_cts = sorted(set.intersection(*map(set, ct_by_contrast.values())))
for c, v in ct_by_contrast.items():
    extra = sorted(set(v) - set(shared_cts))
    print(f'{c}: {len(v)} cell types' + (f'  (only in this contrast: {", ".join(extra)})' if extra else ''))
n_by_ct = de.groupby('cell_type')['gene_name'].nunique()
print(f'{len(n_by_ct)} cell types in total, tested in both contrasts: {len(shared_cts)}')
print(f'genes tested per cell type: {n_by_ct.min():,} - {n_by_ct.max():,} (median {int(n_by_ct.median()):,})')
display(n_by_ct.reindex(ct_order(present=n_by_ct.index)).to_frame('genes tested').T)


2 of 870,930 Wald statistics capped at +/-20


,cell_type,contrast,gene_name,baseMean,stat
629836,TEC-EMT,adult_vs_infant,SCGB2A2,6.103285,3.361936e+07
632220,TEC-EMT,adult_vs_infant,IGHV2-5,6.669867,5.170339e+04


paed_vs_infant: 23 cell types
adult_vs_infant: 26 cell types  (only in this contrast: EC-lymphatic, Mono, Myeloid_dev)


26 cell types in total, tested in both contrasts: 23
genes tested per cell type: 8,722 - 27,510 (median 17,585)


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,EC-lymphatic,SMC,Pericyte,B_cell,B_plasma,Myeloid_dev,Mono,Mac,DC1,DC2,aDC,pDC
genes tested,15279,20652,23382,18467,20913,20087,18483,13505,20012,21699,27510,17274,23646,22126,13490,17897,15498,20005,15027,8722,11482,14048,13352,12979,13273,13504


## 2. Network: SCENIC+ eRegulons of the thymus multiome

eRegulons were inferred with SCENIC+ on the matched thymus single-nucleus multiome data, taking the
consensus over 21 runs and keeping TF–region–gene triplets recovered in at least three runs. Each
eRegulon is named `TF_<TF→gene sign>_<region→gene sign>_(n targets g)`; the network is written in
decoupler format with weight **+1** for activating (`_+_+`) and **−1** for repressing (`_−_+`)
eRegulons. Because the sign is part of an eRegulon's identity every regulon is sign-homogeneous, and
a TF can contribute one activating and one repressing eRegulon with disjoint target sets.

With sign-homogeneous ±1 weights the ULM score is a contrast of the regulon's targets against all
other tested genes, and **a positive score means the TF's activity rises with age for both
classes**: an activating eRegulon scores positive when its targets go up, a repressing one when its
targets go down.


In [6]:
net = pd.read_csv(NET_CSV)
assert list(net.columns) == ['source', 'target', 'weight'], net.columns.tolist()
assert set(net['weight'].unique()) <= {-1.0, 1.0}, 'weights are not sign-only'
net_sources = set(net['source'])

_SRC_RE = re.compile(r'^(.+)_([+-])_([+-])_\((\d+)([gr])\)$')


def parse_source(name):
    """'AR_+_+_(78g)' -> ('AR', '+', '+', 78)."""
    m = _SRC_RE.match(name)
    if not m:
        raise ValueError(f'not an eRegulon name: {name!r}')
    return m.group(1), m.group(2), m.group(3), int(m.group(4))


def annotate_sources(sources):
    rows = [dict(zip(('tf', 'tf2g_sign', 'r2g_sign', 'n_named'), parse_source(s)), source=s) for s in sorted(sources)]
    df = pd.DataFrame(rows)[['source', 'tf', 'tf2g_sign', 'r2g_sign', 'n_named']]
    df['mode'] = np.where(df['tf2g_sign'] == '+', 'activating', 'repressing')
    return df.set_index('source')


ann = annotate_sources(net_sources)
w = net.groupby('source')['weight'].agg(['min', 'max'])
assert (w['min'] == w['max']).all(), 'a regulon mixes weights'
assert all(w.loc[s, 'min'] == (1.0 if ann.loc[s, 'tf2g_sign'] == '+' else -1.0) for s in w.index)
tgt = net.groupby('source')['target'].apply(set)
n_tg = tgt.map(len)
both = ann.groupby('tf')['mode'].nunique()
dual = both.index[both == 2]
shared = [len(tgt[a] & tgt[r]) for tf in dual
          for a, r in [(ann.index[(ann['tf'] == tf) & (ann['mode'] == 'activating')][0],
                        ann.index[(ann['tf'] == tf) & (ann['mode'] == 'repressing')][0])]]
print(f'{len(net_sources)} eRegulons over {ann["tf"].nunique()} TFs: '
      f'{(ann["mode"] == "activating").sum()} activating / {(ann["mode"] == "repressing").sum()} repressing | '
      f'{len(net):,} edges | {net["target"].nunique():,} target genes | '
      f'targets per regulon: median {int(n_tg.median())}, range {n_tg.min()}-{n_tg.max()}')
print(f'{len(dual)} TFs have both an activating and a repressing eRegulon; '
      f'{sum(s > 0 for s in shared)} of them share any target between the two (max shared {max(shared)})')
measured = set(de['gene_name'])
cov = net.assign(measured=net['target'].isin(measured)).groupby('source')['measured'].sum()
print(f'targets measured in the DE: median {int(cov.median())} per eRegulon; '
      f'{int((cov < TMIN).sum())} eRegulons below TMIN={TMIN} atlas-wide')
steroid_regulons = ann[ann['tf'].isin(STEROID_TFS)]
print('\nsteroid-receptor eRegulons in the network:')
display(steroid_regulons.assign(targets_measured=cov.reindex(steroid_regulons.index)))
print('no eRegulon for:', [t for t in STEROID_TFS if t not in set(ann['tf'])])


400 eRegulons over 298 TFs: 286 activating / 114 repressing | 74,660 edges | 10,722 target genes | targets per regulon: median 90, range 5-1733
102 TFs have both an activating and a repressing eRegulon; 0 of them share any target between the two (max shared 0)
targets measured in the DE: median 90 per eRegulon; 0 eRegulons below TMIN=5 atlas-wide

steroid-receptor eRegulons in the network:


,tf,tf2g_sign,r2g_sign,n_named,mode,targets_measured
source,,,,,,
AR_+_+_(78g),AR,+,+,78,activating,78
NR3C1_-_+_(18g),NR3C1,-,+,18,repressing,18


no eRegulon for: ['PGR', 'ESR1', 'ESR2']


## 3. Scoring: univariate linear model on the Wald statistics, calibrated against an abundance-matched permutation null

Every cell type is scored **on its own tested genes** (nothing is discarded to build a gene universe
shared across cell types). For one cell type × contrast:

1. **Adjacency.** A genes × regulons weight matrix over the tested genes (0 for non-targets),
   restricted to regulons with at least `TMIN` measured targets and column-standardised — exactly what
   `decoupler`'s `ulm` builds internally.
2. **Score.** The univariate linear model of decoupler: for each regulon, the Pearson correlation `r`
   between the vector of Wald statistics and the regulon's weight vector, reported as the t-value
   `score = r · sqrt(df / (1 − r²))`, `df = n_genes − 2`. Because the adjacency is fixed for a cell
   type, the score of any vector is a single matrix product, which is what makes 1,000 permutations
   affordable. The next cell checks that this reproduces `dc.mt.ulm` exactly.
3. **Null.** Genes are placed in `N_BINS` quantile bins of `baseMean`, and the Wald statistics are
   permuted *within* bins `N_PERM` times. Each permutation is scored; `null_mean` / `null_sd` are the
   mean and SD of a regulon's score across permutations, i.e. what a random gene set of the same size
   **and the same expression-level profile** scores in that cell type.
4. **Calibrated statistic.** `z = (score − null_mean) / null_sd`, `p_z = 2·Φ(−|z|)`, plus the empirical
   `p_emp`. `padj` is Benjamini–Hochberg across cell types within each regulon × contrast.
5. **Percentile.** `pct_z` is the percentile rank of a regulon's `z` among all regulons of the network
   scoreable in that cell type × contrast (100 = the regulon with the highest `z`). Because the ULM
   t-value grows with `sqrt(n_genes)` and the permutation does not remove that scaling, `z` is
   comparable *within* a cell type but not between cell types with different gene counts; the
   percentile (or `r`, which is scale-free) is what to compare across populations.


In [7]:
EPS = 2.2e-16                     # decoupler's guard in its t-value


def build_adjacency(universe, net, tmin=TMIN):
    """Genes x regulons weight matrix over `universe` (sorted gene symbols), column-standardised.
    Regulons with fewer than `tmin` targets in the universe are dropped (decoupler's `tmin`)."""
    gi = pd.Series(np.arange(len(universe)), index=universe)
    sub = net[net['target'].isin(gi.index)]
    n_tg = sub.groupby('source')['target'].size()
    srcs = sorted(n_tg.index[n_tg >= tmin])
    sub = sub[sub['source'].isin(set(srcs))]
    si = pd.Series(np.arange(len(srcs)), index=srcs)
    A = np.zeros((len(universe), len(srcs)), dtype=np.float64)
    A[gi[sub['target']].to_numpy(), si[sub['source']].to_numpy()] = sub['weight'].to_numpy()
    A_std = (A - A.mean(axis=0)) / A.std(axis=0, ddof=1)
    return srcs, A_std, n_tg.reindex(srcs).to_numpy()


def ulm(Y, A_std):
    """decoupler's univariate linear model written out: every row of Y (a vector over the universe)
    against every column of A_std. Returns (t, r) with t = r*sqrt(df/(1-r^2)), df = n_genes - 2."""
    Y = np.asarray(Y, dtype=np.float64)          # float64: float32 accumulation drifts ~2e-4
    n = Y.shape[1]
    Y_std = (Y - Y.mean(axis=1, keepdims=True)) / Y.std(axis=1, ddof=1, keepdims=True)
    r = (Y_std @ A_std) / (n - 1)
    return r * np.sqrt((n - 2) / ((1.0 - r + EPS) * (1.0 + r + EPS))), r


def abundance_bins(basemean, n_bins=N_BINS):
    """Quantile bin of baseMean per gene (ranked first so ties split deterministically)."""
    return pd.qcut(pd.Series(basemean).rank(method='first'), n_bins, labels=False).to_numpy()


def permutation_null(y, basemean, A_std, n_perm=N_PERM, seed=SEED, batch=200):
    """n_perm x n_regulons scores of `y` permuted WITHIN abundance bins (a regulon keeps its
    targets; the statistics are shuffled among genes of similar expression level)."""
    bins = abundance_bins(basemean)
    order = np.argsort(bins, kind='stable')
    g = bins[order].astype(np.float64)
    y_sorted = np.asarray(y, dtype=np.float64)[order]
    rng = np.random.default_rng(seed)
    out = np.empty((n_perm, A_std.shape[1]), dtype=np.float32)
    for start in range(0, n_perm, batch):
        k = min(batch, n_perm - start)
        keys = g[None, :] + rng.random((k, g.size))      # sorting keeps each bin's block intact
        out[start:start + k] = ulm(y_sorted[np.argsort(keys, axis=1, kind='stable')], A_std)[0]
    return out


def score_cell_type(d_ct, net, contrasts, n_perm=N_PERM, drop_genes=None):
    """All regulons in one cell type (the rows of `de` for it), on its own tested genes, per contrast."""
    ct = d_ct['cell_type'].iloc[0]
    universe = np.sort(d_ct['gene_name'].unique())
    if drop_genes is not None:                           # remove genes from the universe entirely
        universe = universe[~np.isin(universe, np.array(sorted(drop_genes)))]
    srcs, A_std, n_tg = build_adjacency(universe, net)
    rows = []
    for contrast in contrasts:
        d = d_ct[d_ct['contrast'] == contrast]
        if d.empty:
            continue
        d = d.set_index('gene_name').reindex(universe)
        y = d['stat'].to_numpy()
        t_obs, r_obs = ulm(y[None, :], A_std)
        null = permutation_null(y, d['baseMean'].to_numpy(), A_std, n_perm=n_perm)
        mu, sd = null.mean(axis=0), null.std(axis=0, ddof=1)
        z = (t_obs[0] - mu) / np.where(sd > 0, sd, np.nan)
        p_emp = (1 + (np.abs(null - mu) >= np.abs(t_obs[0] - mu)).sum(axis=0)) / (n_perm + 1)
        rows.append(pd.DataFrame({'cell_type': ct, 'contrast': contrast, 'source': srcs,
                                  'n_genes': len(universe), 'n_targets': n_tg,
                                  'score': t_obs[0], 'r': r_obs[0], 'null_mean': mu, 'null_sd': sd,
                                  'z': z, 'p_z': 2 * norm.sf(np.abs(z)), 'p_emp': p_emp}))
    return pd.concat(rows, ignore_index=True)


def score_all(de, net, contrasts=CONTRASTS, cell_types=None, n_perm=N_PERM, drop_genes=None):
    """Every cell type x contrast; `padj` = Benjamini-Hochberg across cell types within regulon x contrast."""
    cts = sorted(de['cell_type'].unique()) if cell_types is None else list(cell_types)
    out, t0 = [], time.time()
    for i, ct in enumerate(cts, 1):
        r = score_cell_type(de[de['cell_type'] == ct], net, contrasts, n_perm, drop_genes)
        out.append(r)
        print(f'[{i}/{len(cts)}] {ct}: {r["n_genes"].iloc[0]:,} genes, {r["source"].nunique()} regulons '
              f'({time.time() - t0:.0f} s)', flush=True)
    res = pd.concat(out, ignore_index=True)
    res['padj'] = (res.groupby(['source', 'contrast'], observed=True)['p_z']
                      .transform(lambda p: multipletests(p, method='fdr_bh')[1]))
    return res


def add_percentiles(res, pool, by=('cell_type', 'contrast'), suffix=''):
    """`pct_z` / `pct_score`: percentile rank (100 = top) of z / raw score among the regulons of
    `pool` within each cell type x contrast; `n_pool` = number of regulons ranked against."""
    m = res['source'].isin(pool)
    g = res[m].groupby(list(by), observed=True)
    res.loc[m, 'pct_z' + suffix] = g['z'].rank(pct=True) * 100
    res.loc[m, 'pct_score' + suffix] = g['score'].rank(pct=True) * 100
    res.loc[m, 'n_pool' + suffix] = g['z'].transform('size')
    return res


print(f'TMIN={TMIN} | |stat| cap {STAT_CAP:g} | {N_BINS} abundance bins | {N_PERM} permutations | seed {SEED}')


TMIN=5 | |stat| cap 20 | 20 abundance bins | 1000 permutations | seed 0


In [8]:
def check_against_decoupler(de, net, cell_types, contrast='adult_vs_infant'):
    """The matrix-product ULM must reproduce `dc.mt.ulm` on the observed statistics."""
    for ct in cell_types:
        d = de[(de['cell_type'] == ct) & (de['contrast'] == contrast)].sort_values('gene_name')
        universe = d['gene_name'].to_numpy()
        srcs, A_std, _ = build_adjacency(universe, net)
        mine = ulm(d['stat'].to_numpy()[None, :], A_std)[0][0]
        M = pd.DataFrame([d['stat'].to_numpy()], index=[ct], columns=universe)
        theirs = dc.mt.ulm(data=M, net=net, tmin=TMIN, empty=False)[0].reindex(columns=srcs)
        np.testing.assert_allclose(mine, theirs.to_numpy()[0], rtol=1e-6, atol=1e-6)
        print(f'{ct}: reproduces dc.mt.ulm over {len(srcs)} regulons and {len(universe):,} genes')


# checked at both ends of the gene-count range, i.e. the smallest and the largest universe
check_against_decoupler(de, net, [n_by_ct.idxmin(), n_by_ct.idxmax()])


Myeloid_dev: reproduces dc.mt.ulm over 350 regulons and 8,722 genes


mTEC: reproduces dc.mt.ulm over 400 regulons and 27,510 genes


## 4. Score every cell type → Supplementary Table

Same columns as the CollecTRI table (`score`, `r`, `null_mean`, `null_sd`, `z`, `p_z`, `p_emp`,
`padj`, `pct_z`, `pct_score`, `n_pool`; see notebook 01), plus:

| column | meaning |
| --- | --- |
| `tf` | the transcription factor of the eRegulon |
| `mode` | `activating` (`_+_+`) or `repressing` (`_−_+`) |
| `pct_z_mode` | percentile of `z` among the eRegulons **of the same class** in that cell type × contrast — the column to rank on (section 4b) |

`pct_z` ranks against all 400 eRegulons and is kept for completeness.


In [9]:
TABLE_COLS = ['cell_type', 'contrast', 'source', 'tf', 'mode', 'n_genes', 'n_targets', 'score', 'r',
              'null_mean', 'null_sd', 'z', 'p_z', 'p_emp', 'padj', 'pct_z', 'pct_score', 'n_pool', 'pct_z_mode']
if FORCE_RESCORE or not os.path.exists(SCORES_CSV):
    res = score_all(de, net)
    res = add_percentiles(res, pool=net_sources)
    res['tf'] = res['source'].map(ann['tf'])
    res['mode'] = res['source'].map(ann['mode'])
    res['pct_z_mode'] = res.groupby(['cell_type', 'contrast', 'mode'], observed=True)['z'].rank(pct=True) * 100
    res = res[TABLE_COLS]
    res.to_csv(SCORES_CSV, index=False)
    print('written', SCORES_CSV)
else:
    res = pd.read_csv(SCORES_CSV)
    print('loaded cached scores from', SCORES_CSV)
print(f'{len(res):,} rows | {res["source"].nunique()} eRegulons | {res["cell_type"].nunique()} cell types')
ex_ct = ct_order(lineages=['TEC'], present=res['cell_type'])[0]
print(f'\nexample: the steroid-receptor eRegulons in {ex_ct}, adult vs infant')
display(res[res['tf'].isin(STEROID_TFS) & (res['cell_type'] == ex_ct) & (res['contrast'] == 'adult_vs_infant')]
        .set_index('source').round(3))


loaded cached scores from /nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/notebooks/hormone2cell/_builders/../final_versions/github/results/thyAgeing_SCENICplus_ageDE_taa_l2_v2_scores_perCellType.csv
19,176 rows | 400 eRegulons | 26 cell types

example: the steroid-receptor eRegulons in cTEC, adult vs infant


,cell_type,contrast,tf,mode,n_genes,n_targets,score,r,null_mean,null_sd,z,p_z,p_emp,padj,pct_z,pct_score,n_pool,pct_z_mode
source,,,,,,,,,,,,,,,,,,
AR_+_+_(78g),cTEC,adult_vs_infant,AR,activating,20012,71,4.509,0.032,0.333,0.940,4.443,0.0,0.001,0.0,82.915,82.915,398.0,76.491
NR3C1_-_+_(18g),cTEC,adult_vs_infant,NR3C1,repressing,20012,18,-4.822,-0.034,-0.059,1.021,-4.665,0.0,0.001,0.0,8.794,9.045,398.0,17.699


In [10]:
# Null QC: `null_sd` should sit near 1 for every regulon (a t-value of a permuted vector is ~N(0,1)
# whatever the gene count), so what the permutation supplies is the CENTRE: `null_mean` is where an
# abundance-matched random gene set scores, and it varies by regulon and cell type.
print('null_sd  :', res['null_sd'].describe(percentiles=[.01, .5, .99]).round(3)[['1%', '50%', '99%']].to_dict())
print('null_mean:', res['null_mean'].describe(percentiles=[.01, .5, .99]).round(3)[['1%', '50%', '99%']].to_dict())
n_src = res.groupby(['cell_type', 'contrast'], observed=True).size()
print(f'regulons scoreable per cell type x contrast: {n_src.min()} - {n_src.max()} (median {int(n_src.median())})')
summary = res.groupby('contrast', observed=True).apply(
    lambda d: pd.Series({'cell types': d['cell_type'].nunique(), 'rows': len(d),
                         'padj < 0.05': int((d['padj'] < 0.05).sum()),
                         'fraction z > 0': round(float((d['z'] > 0).mean()), 3)}), include_groups=False)
display(summary)


null_sd  : {'1%': 0.88, '50%': 1.031, '99%': 1.175}
null_mean: {'1%': -2.612, '50%': 0.017, '99%': 2.251}
regulons scoreable per cell type x contrast: 350 - 400 (median 395)


,cell types,rows,padj < 0.05,fraction z > 0
contrast,,,,
adult_vs_infant,26.0,10144.0,5155.0,0.528
paed_vs_infant,23.0,9032.0,3762.0,0.531


In [11]:
# Reproducibility check against the scoring run used for the manuscript (internal path; skipped
# when the file is not available).
REF_CSV = ('/nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/results/scRNAseq/regulons_dea_v1/'
           f'thyAgeing_scRNAseq_regulonDEA_perCellType_{LEVEL}.csv')
if os.path.exists(REF_CSV) and N_PERM == 1000:
    ref = pd.read_csv(REF_CSV)
    m = res.merge(ref, on=['cell_type', 'contrast', 'source'], suffixes=('', '_ref'))
    print(f'{len(m):,} of {len(res):,} rows matched to the reference table')
    for col in ['score', 'r', 'null_mean', 'null_sd', 'z', 'pct_z']:
        print(f'  {col:10s} max |difference| = {np.nanmax(np.abs(m[col] - m[col + "_ref"])):.2e}')
    np.testing.assert_allclose(m['z'], m['z_ref'], atol=1e-3, rtol=1e-4)
    print('identical to the reference run')
else:
    print('reference table not available (or N_PERM != 1000) - check skipped')


19,176 of 19,176 rows matched to the reference table
  score      max |difference| = 1.12e-13
  r          max |difference| = 9.02e-16
  null_mean  max |difference| = 0.00e+00
  null_sd    max |difference| = 0.00e+00
  z          max |difference| = 1.12e-13
  pct_z      max |difference| = 0.00e+00
identical to the reference run


### 4a. Direction check against known thymic ageing biology

Everything downstream inherits the sign convention, so it is checked on regulons whose direction is
known independently: the cortical-identity factor FOXN1 and the lipid-synthesis factor SREBF1 lose
activity in ageing TECs, whereas CEBPB / CEBPD (the adipogenic–inflammatory module) gain it.


In [12]:
tec_ct = ct_order(lineages=['TEC'], present=res['cell_type'])
expect = {'FOXN1': 'down', 'SREBF1': 'down', 'CEBPB': 'up', 'CEBPD': 'up'}
rows = []
for tf, direction in expect.items():
    for src in ann.index[ann['tf'] == tf]:
        d = res[(res['source'] == src) & res['cell_type'].isin(tec_ct) & (res['contrast'] == 'adult_vs_infant')]
        if d.empty:
            continue
        med = d['z'].median()
        rows.append({'tf': tf, 'eRegulon': src, 'expected': direction, 'median_z_TEC': round(med, 2),
                     'n_TEC_populations': len(d), 'FDR<0.05': int((d['padj'] < 0.05).sum()),
                     'agrees': bool(med < 0) if direction == 'down' else bool(med > 0)})
check = pd.DataFrame(rows)
display(check)
print(f'adult vs infant, TEC ({", ".join(tec_ct)}): {int(check["agrees"].sum())}/{len(check)} regulons move as expected')


,tf,eRegulon,expected,median_z_TEC,n_TEC_populations,FDR<0.05,agrees
0,FOXN1,FOXN1_+_+_(23g),down,-8.39,4,4,True
1,SREBF1,SREBF1_+_+_(45g),down,-15.22,4,4,True
2,CEBPB,CEBPB_+_+_(467g),up,11.02,4,4,True
3,CEBPB,CEBPB_-_+_(26g),up,-1.05,4,0,False
4,CEBPD,CEBPD_+_+_(1733g),up,15.69,4,4,True
5,CEBPD,CEBPD_-_+_(263g),up,1.48,4,2,True


adult vs infant, TEC (cTEC, mcTEC, mTEC, TEC-EMT): 5/6 regulons move as expected


### 4b. The class offset, and why regulons are ranked within their class

eRegulon target genes are systematically **more abundant** than non-target genes (SCENIC+ can only
call a link for a well-detected gene) and, in most cell types, drift **upward** with age relative to
non-targets by more than the abundance-matched null removes. One drift has opposite consequences
for the two classes: an activating regulon scores positive when its targets rise, a repressing one
scores negative. Activating regulons therefore sit above zero and repressing ones below it
irrespective of biology, and a ranking on `z` or on `pct_z` would order regulons by class first. The
cells below measure the drift and the offset; `pct_z_mode` (percentile within the regulon's own
class) is the ranking used from here on.


In [13]:
act = set(net.loc[net['source'].isin(ann.index[ann['mode'] == 'activating']), 'target'])
rep = set(net.loc[net['source'].isin(ann.index[ann['mode'] == 'repressing']), 'target'])
rows = []
for (ct, c), d in de.groupby(['cell_type', 'contrast'], observed=True):
    isa, isr = d['gene_name'].isin(act), d['gene_name'].isin(rep)
    non = ~(isa | isr)
    rows.append({'cell_type': ct, 'contrast': c,
                 'mean_stat_targets': d.loc[isa | isr, 'stat'].mean(), 'mean_stat_nonTargets': d.loc[non, 'stat'].mean(),
                 'median_baseMean_targets': d.loc[isa | isr, 'baseMean'].median(),
                 'median_baseMean_nonTargets': d.loc[non, 'baseMean'].median()})
drift = pd.DataFrame(rows)
drift['drift'] = drift['mean_stat_targets'] - drift['mean_stat_nonTargets']
drift.to_csv(f'{RESULTS}/{PREFIX}_targetDrift.csv', index=False)
print(f'targets vs non-targets: median baseMean {drift["median_baseMean_targets"].median():.2f} vs '
      f'{drift["median_baseMean_nonTargets"].median():.2f}; mean Wald-statistic difference median '
      f'{drift["drift"].median():+.3f}, positive in {int((drift["drift"] > 0).sum())} of {len(drift)} cell type x contrast strata')

by_mode = res.groupby(['contrast', 'mode'], observed=True)['z'].agg(
    n='size', median_z='median', fraction_positive=lambda s: float((s > 0).mean()))
print('\nwhere each class sits relative to zero:')
display(by_mode.round(3))


targets vs non-targets: median baseMean 14.73 vs 1.63; mean Wald-statistic difference median +0.163, positive in 43 of 49 cell type x contrast strata

where each class sits relative to zero:


n  median_z  fraction_positive
contrast        mode                                         
adult_vs_infant activating  7210     1.016              0.638
                repressing  2934    -1.874              0.258
paed_vs_infant  activating  6429     0.940              0.664
                repressing  2603    -1.682              0.202

## 5. The steroid-receptor eRegulons among all eRegulons

Only **AR** (activating eRegulon) and **NR3C1** (repressing eRegulon) have an eRegulon in the thymus
network; PGR, ESR1 and ESR2 do not, so they cannot be scored here (they are in the CollecTRI
notebook). The same two views as in notebook 01: the regulons within the per-cell-type distribution
of `z` over all eRegulons (violins; filled = FDR < 0.05), and their percentiles across populations.
Percentiles are shown **within class** (`pct_z_mode`), with the whole-network `pct_z` in the table.


In [14]:
all_ct    = ct_order(present=res['cell_type'])
stroma_ct = ct_order(lineages=STROMA_L1, present=res['cell_type'])
print(f'{len(all_ct)} cell types; {len(stroma_ct)} stromal: {", ".join(stroma_ct)}')

srcs = list(steroid_regulons.index)
LABEL = {s: f'{ann.loc[s, "tf"]} ({ann.loc[s, "mode"]}, {ann.loc[s, "n_named"]} targets)' for s in srcs}
rec = res[res['source'].isin(srcs)].copy()
rec.to_csv(f'{RESULTS}/{PREFIX}_steroidReceptors_table.csv', index=False)
for contrast in CONTRASTS:
    sub = rec[rec['contrast'] == contrast]
    here = [c for c in all_ct if c in set(sub['cell_type'])]
    for col in ['pct_z_mode', 'pct_z']:
        piv = sub.pivot(index='source', columns='cell_type', values=col).reindex(srcs)[here]
        sig = sub.pivot(index='source', columns='cell_type', values='padj').reindex(srcs)[here] < 0.05
        print(f'\n{contrast} - {col}; * = FDR < 0.05')
        display(piv.map(lambda v: f'{v:.0f}' if np.isfinite(v) else '') + sig.map(lambda s: '*' if s else ''))
med = (rec[rec['cell_type'].isin(stroma_ct)].groupby(['source', 'contrast'])[['pct_z_mode', 'pct_z']].median()
       .unstack('contrast').reindex(srcs))
print('\nmedian percentiles across the stromal populations:')
display(med.round(1))


26 cell types; 9 stromal: cTEC, mcTEC, mTEC, TEC-EMT, Fb, EC-blood, EC-lymphatic, SMC, Pericyte

paed_vs_infant - pct_z_mode; * = FDR < 0.05


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,SMC,Pericyte,B_cell,B_plasma,Mac,DC1,DC2,aDC,pDC
source,,,,,,,,,,,,,,,,,,,,,,,
AR_+_+_(78g),32,13,14,14,23,23,19,38,70*,59,52,80*,89*,19,7*,74*,20,56,51,31,16,4,11
NR3C1_-_+_(18g),11*,51*,76,88,90,68,86,34*,7*,12*,8*,18*,10*,18,64,46*,74,8*,25*,51,72,27*,29*



paed_vs_infant - pct_z; * = FDR < 0.05


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,SMC,Pericyte,B_cell,B_plasma,Mac,DC1,DC2,aDC,pDC
source,,,,,,,,,,,,,,,,,,,,,,,
AR_+_+_(78g),49,31,27,31,37,37,34,54,78*,69,58,84*,90*,17,10*,79*,31,64,62,48,35,18,31
NR3C1_-_+_(18g),3*,18*,45,51,59,33,51,12*,5*,9*,6*,11*,10*,21,35,19*,44,8*,8*,21,30,9*,9*



adult_vs_infant - pct_z_mode; * = FDR < 0.05


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,EC-lymphatic,SMC,Pericyte,B_cell,B_plasma,Myeloid_dev,Mono,Mac,DC1,DC2,aDC,pDC
source,,,,,,,,,,,,,,,,,,,,,,,,,,
AR_+_+_(78g),38,46,46,15,13,23,14,45,76*,62,79*,89*,91*,69,54,74*,87*,26*,68*,44,36,53,10,10,47,30
NR3C1_-_+_(18g),42*,40*,80,88,58,53,51*,38*,18*,10*,26*,20*,21*,37,47,42,44,55,14*,2*,37*,65,88,95,73,99



adult_vs_infant - pct_z; * = FDR < 0.05


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,EC-lymphatic,SMC,Pericyte,B_cell,B_plasma,Myeloid_dev,Mono,Mac,DC1,DC2,aDC,pDC
source,,,,,,,,,,,,,,,,,,,,,,,,,,
AR_+_+_(78g),54,59,59,26,25,35,30,55,83*,71,85*,91*,94*,76,62,81*,91*,30*,72*,54,49,66,30,29,58,45
NR3C1_-_+_(18g),16*,15*,33,76,29,25,21*,20*,9*,11*,18*,17*,12*,29,28,25,31,54,12*,1*,14*,29,37,53,46,72



median percentiles across the stromal populations:


pct_z_mode                          pct_z               
contrast        adult_vs_infant paed_vs_infant adult_vs_infant paed_vs_infant
source                                                                       
AR_+_+_(78g)               76.5           64.6            82.9           73.7
NR3C1_-_+_(18g)            26.3           15.0            17.8           10.3

In [15]:
def _robust_lims(values, markers, q=(0.002, 0.998), pad=0.05):
    v = values[np.isfinite(values)]
    lo, hi = np.nanquantile(v, q)
    m = markers[np.isfinite(markers)]
    if m.size:                                           # never clip a highlighted marker out
        lo, hi = min(lo, m.min()), max(hi, m.max())
    d = pad * (hi - lo)
    return lo - d, hi + d


def plot_tf_violins(res, cts, style, value='z', padj_col='padj', contrasts=CONTRASTS,
                    width_mm='auto', panel_h_mm=38.0, per_ct_mm=6.0, pad_mm=18.0,
                    ylabel='z vs abundance-matched null', source_label='TFs', title=None,
                    marker_pt=2.8, point_pt=0.6):
    """Where the highlighted regulons sit in the distribution of ALL regulons of each cell type.

    Cell types on x, one panel per contrast (stacked, shared axes). Per cell type: a violin and a
    rug of `value` over every regulon scoreable there; the highlighted regulons (`style` =
    {source: (colour, marker, label)}) drawn on top, filled when `padj_col` < 0.05. Cell types not
    tested in a contrast are marked "n/t". Sized in mm: `width_mm='auto'` = pad + per_ct_mm per cell
    type, capped at Nature's double-column width.
    """
    present = [c for c in cts if c in set(res['cell_type'])]
    srcs = [s for s in style if s in set(res['source'])]
    if width_mm == 'auto':
        width_mm = float(np.clip(pad_mm + per_ct_mm * len(present), 45.0, NATURE_W['double']))
    key_labels = [style[s][2] for s in srcs] + ['FDR < 0.05', 'n.s.', f'all {source_label}']
    n_keys = len(key_labels)

    def _legend_mm(ncol):                                # fig.legend does not wrap: size the columns
        rows = int(np.ceil(n_keys / ncol))              # from the labels (column-major fill)
        cols = [key_labels[i * rows:(i + 1) * rows] for i in range(ncol)]
        return sum(1.1 * max(len(l) for l in c) + 7.0 for c in cols if c)

    ncol = max([n for n in range(1, n_keys + 1) if _legend_mm(n) <= width_mm - 6.0] or [1])
    legend_rows = int(np.ceil(n_keys / ncol))
    lab_mm = 1.25 * max(len(str(c)) for c in present) + 3.0
    h_mm = len(contrasts) * panel_h_mm + lab_mm + 10.0 + 5.0 * legend_rows
    fig = mm_fig(width_mm, h_mm)
    axes = np.atleast_1d(fig.subplots(len(contrasts), 1, sharex=True, sharey=True))
    pos = {c: i for i, c in enumerate(present)}

    for ax, contrast in zip(axes, contrasts):
        sub = res[res['contrast'] == contrast]
        data, at = [], []
        for c in present:
            v = sub.loc[sub['cell_type'] == c, value].to_numpy()
            v = v[np.isfinite(v)]
            if v.size:
                data.append(v)
                at.append(pos[c])
                ax.scatter(np.full(v.size, pos[c]), v, s=point_pt ** 2, c='0.35', alpha=0.6,
                           linewidths=0, zorder=2, rasterized=True)
            else:
                ax.text(pos[c], 0, 'n/t', ha='center', va='center', fontsize=6, color='0.5')
        parts = ax.violinplot(data, positions=at, showextrema=False, widths=0.86)
        for pc in parts['bodies']:
            pc.set_facecolor('#dedede'); pc.set_edgecolor('#9e9e9e'); pc.set_linewidth(0.4)
            pc.set_alpha(0.6); pc.set_zorder(1)
        ax.axhline(0, color='0.55', lw=0.5, zorder=0)
        for c in present:
            d = sub[sub['cell_type'] == c].set_index('source')
            for s in srcs:
                if s not in d.index or not np.isfinite(d.loc[s, value]):
                    continue
                col, mk, _ = style[s]
                sig = d.loc[s, padj_col] < 0.05
                ax.scatter(pos[c], d.loc[s, value], marker=mk,
                           s=(marker_pt * (1.4 if mk == '*' else 1.0)) ** 2, zorder=3,
                           facecolor=(*to_rgb(col), 0.75) if sig else 'none', edgecolor=col,
                           linewidths=0.5)
        n_src = int(sub[sub['cell_type'].isin(present)].groupby('cell_type', observed=True).size().max())
        ax.set_title(f'{contrast.replace("_", " ")}  (\u2264 {n_src:,} {source_label})')
        ax.set_ylabel(ylabel)
        ax.grid(axis='y', color='0.92', lw=0.4)
        ax.set_axisbelow(True)

    sub = res[res['cell_type'].isin(present) & res['contrast'].isin(contrasts)]
    axes[0].set_ylim(*_robust_lims(sub[value].to_numpy(), sub.loc[sub['source'].isin(srcs), value].to_numpy()))
    axes[-1].set_xticks(range(len(present)))
    axes[-1].set_xticklabels(present, rotation=90)
    axes[-1].set_xlim(-0.7, len(present) - 0.3)

    key = 4.5
    handles = [Line2D([], [], color=style[s][0], marker=style[s][1], linestyle='none',
                      markersize=key * (1.4 if style[s][1] == '*' else 1.0), label=style[s][2])
               for s in srcs]
    handles += [Line2D([], [], color='0.3', marker='o', linestyle='none', markersize=key, label='FDR < 0.05'),
                Line2D([], [], color='0.3', marker='o', linestyle='none', markersize=key,
                       markerfacecolor='none', label='n.s.'),
                Line2D([], [], color='0.35', marker='o', linestyle='none', markersize=2, alpha=0.6,
                       label=f'all {source_label}')]
    fig.legend(handles=handles, frameon=False, ncol=ncol, loc='outside lower center',
               handletextpad=0.3, columnspacing=1.0)
    if title:
        fig.suptitle(title, wrap=True)
    return fig


def plot_pct_strip(res, sources, cts, pct_col='pct_z', labels=None, contrasts=CONTRASTS,
                   width_mm=NATURE_W['single'], panel_h_mm=42.0, title=None, ylabel=None,
                   colors=None):
    """Percentile of each highlighted regulon across cell types: regulons on x, percentile on y,
    one point per cell type (coloured by lineage), the median as a bar, one panel per contrast."""
    labels = labels or {}
    lin_present = list(dict.fromkeys(LINEAGE.get(c, '?') for c in cts))
    lab_mm = 1.25 * max(len(labels.get(s, s)) for s in sources) + 2.0
    n_keys = len(lin_present)
    ncol = max(1, min(n_keys, int(width_mm // 20)))
    h_mm = panel_h_mm + lab_mm + 8.0 + 5.0 * int(np.ceil(n_keys / ncol))
    fig = mm_fig(width_mm, h_mm)
    axes = np.atleast_1d(fig.subplots(1, len(contrasts), sharey=True))
    order = {c: i for i, c in enumerate(cts)}
    for ax, contrast in zip(axes, contrasts):
        sub = res[(res['contrast'] == contrast) & res['cell_type'].isin(cts) & res['source'].isin(sources)]
        for i, s in enumerate(sources):
            d = sub[sub['source'] == s].copy()
            d = d[np.isfinite(d[pct_col])]
            if d.empty:
                ax.text(i, 50, 'n/a', ha='center', va='center', fontsize=6, color='0.5')
                continue
            vals = d[pct_col].to_numpy()
            if len(vals) >= 3:
                parts = ax.violinplot([vals], positions=[i], showextrema=False, widths=0.8)
                for pc in parts['bodies']:
                    pc.set_facecolor('#e6e6e6'); pc.set_edgecolor('none'); pc.set_alpha(0.9); pc.set_zorder(1)
            d['x'] = i + np.linspace(-0.26, 0.26, len(d))[np.argsort(np.argsort(d['cell_type'].map(order)))]
            cols = colors if colors is not None else [LINEAGE_COLORS.get(LINEAGE.get(c, '?'), '0.4') for c in d['cell_type']]
            ax.scatter(d['x'], vals, c=cols, s=10, edgecolor='white', linewidths=0.3, zorder=3)
            ax.hlines(np.median(vals), i - 0.32, i + 0.32, color='k', lw=0.9, zorder=4)
        ax.axhline(50, color='0.6', lw=0.5, ls=':', zorder=0)
        ax.set_ylim(-3, 103)
        ax.set_xticks(range(len(sources)))
        ax.set_xticklabels([labels.get(s, s) for s in sources], rotation=90)
        ax.set_xlim(-0.6, len(sources) - 0.4)
        ax.set_title(contrast.replace('_', ' '))
        ax.grid(axis='y', color='0.92', lw=0.4)
        ax.set_axisbelow(True)
    axes[0].set_ylabel(ylabel or f'percentile among all regulons ({pct_col})')
    handles = [Line2D([], [], color=LINEAGE_COLORS.get(l, '0.4'), marker='o', linestyle='none',
                      markersize=4, label=l) for l in lin_present]
    handles += [Line2D([], [], color='k', lw=0.9, label='median')]
    fig.legend(handles=handles, frameon=False, ncol=ncol + 1, loc='outside lower center',
               handletextpad=0.3, columnspacing=1.0)
    if title:
        fig.suptitle(title)
    return fig


In [16]:
style = {s: (*RECEPTOR_STYLE[ann.loc[s, 'tf']], LABEL[s]) for s in srcs}
for scope, cts in [('stroma', stroma_ct), ('allCellTypes', all_ct)]:
    fig = plot_tf_violins(res, cts, style, source_label='eRegulons',
                          title=f'Steroid-receptor eRegulons among all eRegulons - {scope}')
    save_fig(fig, f'{PREFIX}_steroidReceptors_violins_{scope}')
    plt.show()


saved thyAgeing_SCENICplus_ageDE_taa_l2_v2_steroidReceptors_violins_stroma.pdf  (72 x 119 mm)


saved thyAgeing_SCENICplus_ageDE_taa_l2_v2_steroidReceptors_violins_allCellTypes.pdf  (174 x 109 mm)


In [17]:
for scope, cts in [('stroma', stroma_ct), ('allCellTypes', all_ct)]:
    fig = plot_pct_strip(res, srcs, cts, pct_col='pct_z_mode', labels=LABEL, width_mm=NATURE_W['single'],
                         ylabel='percentile within eRegulon class (z)',
                         title=f'Steroid-receptor eRegulon percentiles across {scope} ({len(cts)} populations)')
    save_fig(fig, f'{PREFIX}_steroidReceptors_percentiles_{scope}')
    plt.show()


saved thyAgeing_SCENICplus_ageDE_taa_l2_v2_steroidReceptors_percentiles_stroma.pdf  (89 x 94 mm)


saved thyAgeing_SCENICplus_ageDE_taa_l2_v2_steroidReceptors_percentiles_allCellTypes.pdf  (89 x 104 mm)


## 6. Regulons that move consistently across the stroma: top 20 up and top 20 down

"Shared" is defined by **consistency of rank, not by magnitude**: every regulon is ranked by the
**median of its within-cell-type percentile** over the stromal population × contrast slots. A
percentile is a rank inside one cell type, so it is comparable across populations by construction;
ranking on mean `z` would instead favour the populations with the largest gene universes.

Two filters: a regulon must be scoreable in at least 80% of the slots (a regulon scoreable in only
two populations can trivially reach a median of 99), and must have at least 15 measured targets
(very small regulons are curation artefacts). The figure shows the 20 highest and the 20 lowest
median percentiles as two row blocks, coloured by `z` (diverging, centred on 0, symmetric limits at
the 99th percentile of |z|), with an asterisk where permutation FDR < 0.05, grey where the
population was not tested in that contrast.


In [18]:
def shared_ranking(res, cts, contrasts=CONTRASTS, pct_col='pct_z', min_frac=0.8, min_targets=15):
    """Median within-cell-type percentile over the cell type x contrast slots of `cts`."""
    d = res[res['cell_type'].isin(cts) & res['contrast'].isin(contrasts)]
    n_slots = d.groupby('source').size()
    full = int(n_slots.max())
    d = d[d['source'].isin(n_slots.index[n_slots >= min_frac * full])]
    g = d.groupby('source')
    r = pd.DataFrame({'median_pct': g[pct_col].median(), 'mean_z': g['z'].mean(), 'mean_r': g['r'].mean(),
                      'n_slots': g.size(), 'n_sig': g['padj'].apply(lambda p: int((p < 0.05).sum())),
                      'n_targets': g['n_targets'].max()})
    r['kept'] = r['n_targets'] >= min_targets
    return r.sort_values('median_pct', ascending=False), full


def _matrix(res, sources, cts, contrast, value):
    m = (res[res['contrast'] == contrast]
         .pivot_table(index='source', columns='cell_type', values=value, observed=True))
    return m.reindex(index=sources, columns=cts)


def plot_shared_heatmap(res, cts, up, dn, contrasts=CONTRASTS, value='z', width_mm=NATURE_W['double'],
                        row_mm=3.1, base_pt=7.0, qlim=0.99, star_at=0.05, labels=None, title=None,
                        block_names=('up with age', 'down with age')):
    """Two row blocks (up / down) x cell types, one panel per contrast; colour = `value`."""
    labels = labels or {}
    blocks = [(block_names[0], list(up)), (block_names[1], list(dn))]
    mats = {(b, c): _matrix(res, s, cts, c, value) for b, s in blocks for c in contrasts}
    padj = {(b, c): _matrix(res, s, cts, c, 'padj') for b, s in blocks for c in contrasts}
    v = np.concatenate([m.to_numpy().ravel() for m in mats.values()])
    v = v[np.isfinite(v)]
    vmax = max(float(np.nanquantile(np.abs(v), qlim)), 1e-9)
    n_rows = len(up) + len(dn)
    lab_mm = 1.05 * max(len(str(c)) for c in cts) + 4.0
    h_mm = (n_rows + 1.4) * row_mm + lab_mm + 12.0
    with plt.rc_context({'font.size': base_pt, 'xtick.labelsize': base_pt, 'ytick.labelsize': base_pt,
                         'axes.titlesize': base_pt + 1}):
        fig = mm_fig(width_mm, h_mm)
        gs = fig.add_gridspec(2, len(contrasts) + 1, height_ratios=[len(up), len(dn)],
                              width_ratios=[1] * len(contrasts) + [0.035], hspace=1.4 / n_rows, wspace=0.03)
        for bi, (bname, srcs) in enumerate(blocks):
            for ci, c in enumerate(contrasts):
                ax = fig.add_subplot(gs[bi, ci])
                sns.heatmap(mats[(bname, c)], ax=ax, cmap=DIVERGING, center=0, vmin=-vmax, vmax=vmax,
                            cbar=False, linewidths=0,
                            yticklabels=[labels.get(s, s) for s in srcs] if ci == 0 else False,
                            xticklabels=(bi == 1))
                ax.grid(False)
                ax.set_xlabel('')
                ax.set_ylabel(bname if ci == 0 else '', labelpad=2)
                ax.tick_params(length=1.2, width=0.4, pad=1.2)
                if bi == 0:
                    ax.set_title(c.replace('_', ' '), pad=3)
                if bi == 1:
                    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
                P = padj[(bname, c)]
                for yi in range(P.shape[0]):
                    for xi in range(P.shape[1]):
                        p = P.iat[yi, xi]
                        if np.isfinite(p) and p < star_at:
                            ax.text(xi + 0.5, yi + 0.62, '*', ha='center', va='center', fontsize=base_pt - 1.5)
        cax = fig.add_subplot(gs[:, -1])
        fig.colorbar(ScalarMappable(Normalize(-vmax, vmax), DIVERGING), cax=cax,
                     label='z vs abundance-matched null')
        cax.tick_params(length=1.5, width=0.4, pad=1.5)
        if title:
            fig.suptitle(title)
    return fig


In [19]:
ranking, n_slots_full = shared_ranking(res, stroma_ct, pct_col='pct_z_mode', min_frac=0.8, min_targets=15)
ranking['tf'] = ranking.index.map(ann['tf'])
ranking['mode'] = ranking.index.map(ann['mode'])
ranking.to_csv(f'{RESULTS}/{PREFIX}_sharedRanking_stroma.csv')
ok = ranking[ranking['kept']]
up20, dn20 = list(ok.head(20).index), list(ok.tail(20).index[::-1])
print(f'{len(stroma_ct)} stromal populations = {n_slots_full} population x contrast slots; '
      f'{len(ranking)} eRegulons scoreable in >= 80% of them, {len(ok)} with >= 15 measured targets')

fig = plot_shared_heatmap(res, stroma_ct, up20, dn20,
                          title='SCENIC+ eRegulons most consistently up / down with age across the stroma')
save_fig(fig, f'{PREFIX}_top20_sharedStroma')
plt.show()

shared20 = ranking.loc[up20 + dn20].assign(direction=['up'] * len(up20) + ['down'] * len(dn20))
shared20.to_csv(f'{RESULTS}/{PREFIX}_top20_sharedStroma.csv')
cols = ['direction', 'tf', 'mode', 'median_pct', 'mean_z', 'mean_r', 'n_sig', 'n_slots', 'n_targets']
display(shared20[cols].round(2))
print('class mix - UP:', shared20.loc[shared20['direction'] == 'up', 'mode'].value_counts().to_dict(),
      '| DOWN:', shared20.loc[shared20['direction'] == 'down', 'mode'].value_counts().to_dict())

# the same question asked of the literature network (notebook 01), if its result is present
ct_path = f'{RESULTS}/thyAgeing_CollecTRI_ageDE_{LEVEL}_top20_sharedStroma.csv'
if os.path.exists(ct_path):
    ct20 = pd.read_csv(ct_path, index_col=0)
    for d in ('up', 'down'):
        sc_tfs = set(shared20.loc[shared20['direction'] == d, 'tf'])
        ct_tfs = set(ct20.index[ct20['direction'] == d])
        print(f'{d.upper():4s}: TFs in both top-20 lists (SCENIC+ and CollecTRI): {sorted(sc_tfs & ct_tfs) or "none"}')


9 stromal populations = 17 population x contrast slots; 395 eRegulons scoreable in >= 80% of them, 356 with >= 15 measured targets


saved thyAgeing_SCENICplus_ageDE_taa_l2_v2_top20_sharedStroma.pdf  (183 x 157 mm)


,direction,tf,mode,median_pct,mean_z,mean_r,n_sig,n_slots,n_targets
source,,,,,,,,,
KLF9_+_+_(1063g),up,KLF9,activating,99.30,16.43,0.12,17,17,1063
NFIL3_+_+_(653g),up,NFIL3,activating,98.25,14.02,0.10,16,17,653
FOSL2_+_+_(1048g),up,FOSL2,activating,97.55,14.29,0.11,17,17,1048
CEBPD_+_+_(1733g),up,CEBPD,activating,97.08,11.55,0.08,16,17,1733
NFIB_+_+_(1326g),up,NFIB,activating,96.50,11.81,0.09,16,17,1325
EGR1_+_+_(772g),up,EGR1,activating,96.50,10.89,0.08,15,17,772
KLF4_+_+_(794g),up,KLF4,activating,96.10,11.47,0.08,16,17,794
MAFF_+_+_(292g),up,MAFF,activating,95.99,11.37,0.08,17,17,292
ATF3_+_+_(472g),up,ATF3,activating,95.80,12.13,0.09,17,17,472


class mix - UP: {'activating': 18, 'repressing': 2} | DOWN: {'activating': 13, 'repressing': 7}
UP  : TFs in both top-20 lists (SCENIC+ and CollecTRI): ['CEBPB', 'KLF4', 'STAT3']
DOWN: TFs in both top-20 lists (SCENIC+ and CollecTRI): ['IRF7']


## Outputs

All under `results/` and `figures/` with the prefix `thyAgeing_SCENICplus_ageDE_<level>_`:

**Tables** — `scores_perCellType.csv` (the Supplementary Table), `steroidReceptors_table.csv`,
`targetDrift.csv`, `sharedRanking_stroma.csv`, `top20_sharedStroma.csv`.

**Figures** — `steroidReceptors_violins_{stroma,allCellTypes}.pdf`,
`steroidReceptors_percentiles_{stroma,allCellTypes}.pdf`, `top20_sharedStroma.pdf`.
